In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama

In [7]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
llm = ChatOllama(
    model="qwen2.5:0.5b",
    temperature=0
)

In [12]:
def perguntar(query, k=3):
    results = db.similarity_search(query, k=k)

    contexto = "\n\n".join(
        f"[Trecho {i+1}]\n{doc.page_content}"
        for i, doc in enumerate(results)
    )

    prompt = f"""
Você é um assistente que responde perguntas sobre um livro.

Use apenas os trechos abaixo.
Se a resposta não estiver nos trechos, diga que não encontrou essa informação nos trechos recuperados.

Trechos:
{contexto}

Pergunta:
{query}

Resposta:
"""

    resposta = llm.invoke(prompt)

    print("\nFontes:")
    for i, doc in enumerate(results, start=1):
        page = doc.metadata.get("page")

        if page is not None:
            page = page + 1

        print(f"- Trecho {i}: página {page} | {doc.metadata.get('source')}")
        
    return resposta.content

In [10]:
while True:
    query = input("\nVocê: ")

    if query.lower() in ["sair", "exit", "quit"]:
        break

    resposta = perguntar(query)

    print("\nAssistente:")
    print(resposta)


Fontes:
- Trecho 1: página 431 | C:/Users/otavi/OneDrive/Desktop/virtologia.pdf
- Trecho 2: página 0 | C:/Users/otavi/OneDrive/Desktop/virtologia.pdf
- Trecho 3: página 2 | C:/Users/otavi/OneDrive/Desktop/virtologia.pdf

Assistente:
A Virtologia é uma disciplina de psicologia que aborda as virtudes e suas influências na vida humana. Ela se concentra em entender como as pessoas desenvolvem outras virtudes, como o otimismo, a humildade e a paciência, e como essas virtudes ajudam os indivíduos a construir uma base emocional mais forte e equilibrada.

Fontes:
- Trecho 1: página 92 | C:/Users/otavi/OneDrive/Desktop/mind_over_muscle.pdf
- Trecho 2: página 124 | C:/Users/otavi/OneDrive/Desktop/mind_over_muscle.pdf
- Trecho 3: página 4 | C:/Users/otavi/OneDrive/Desktop/mind_over_muscle.pdf

Assistente:
Não encontrei esse trecho na lista de trechos recuperados. Não encontrou essa informação nos trechos recuperados. Não posso responder a essa pergunta, então não posso fornecer uma resposta corr